# FreeCAD + Ollama (Local LLM) in Google Colab 🚀

This notebook demonstrates how to run an autonomous 3D CAD modeling workflow using **FreeCAD** and a local **Ollama** LLM (e.g., Llama 3 or Mistral) completely within Google Colab.

## 1. Environment Setup
First, we'll install FreeCAD, X virtual framebuffer (`xvfb`) for headless execution, and python 3D rendering packages like `trimesh`.

In [ ]:
!sudo apt update
!sudo add-apt-repository ppa:freecad-maintainers/freecad-stable -y
!sudo apt update
!sudo DEBIAN_FRONTEND=noninteractive apt install -y freecad xvfb npm
!npm install -g obj2gltf
!pip install trimesh pyrender numpy scipy pillow xvfbwrapper pyglet matplotlib

## 2. Setting up the Local LLM (Ollama)
We download and install Ollama, then run it in the background to serve as our agentic reasoning engine.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start Ollama server in the background
process = subprocess.Popen(['ollama', 'serve'])
time.sleep(3)  # Wait for it to start

# Pull a lightweight model (e.g., mistral or llama3)
!ollama pull mistral

## 3. Interpreting User Instructions
We provide the LLM with the context of FreeCAD's Python API, followed by the user's instructions. The LLM translates the user's intent into executable Python code.

In [ ]:
import requests
import json

def generate_freecad_script(user_instruction):
    system_prompt = """
    You are a FreeCAD Python API expert. 
    Write a Python script that uses the `FreeCAD` and `Part` modules to generate the requested 3D model. 
    Ensure you append `/usr/lib/freecad/lib` and `/usr/share/freecad/Mod` to `sys.path`.
    Export the result to 'model.stl' and 'model.obj'.
    Return ONLY valid Python code, no markdown wrappers, no explanations.
    """
    
    prompt = f"{system_prompt}\n\nUser Instruction: {user_instruction}\n\nPython Code:"
    
    response = requests.post("http://localhost:11434/api/generate", 
                             json={"model": "mistral", "prompt": prompt, "stream": False})
    
    if response.status_code == 200:
        return response.json()["response"]
    else:
        return f"Error: {response.status_code}"

# The user's natural language request:
user_instruction = "Create a 3D tree. The trunk should be a cylinder with radius 5 and height 50. Make sure the tree is standing upright on the ground along the Z-axis (gravity axis) so it is not lying flat. The foliage should consist of three spheres on top of the trunk."

generated_code = generate_freecad_script(user_instruction)
print("--- Generated Python Script ---")
print(generated_code)
print("-------------------------------")

# For the sake of this example, if the LLM output is imperfect, here is a working fallback script:
fallback_script = """import sys
import os
sys.path.append("/usr/lib/freecad/lib")
sys.path.append("/usr/share/freecad/Mod")
import FreeCAD
import Part
import Mesh

doc = FreeCAD.newDocument("TreeModel")

# Trunk
trunk = doc.addObject("Part::Cylinder", "Trunk")
trunk.Radius = 5
trunk.Height = 50

# Foliage
foliage1 = doc.addObject("Part::Sphere", "Foliage1")
foliage1.Radius = 25
foliage1.Placement.Base = FreeCAD.Vector(0, 0, 50)
foliage2 = doc.addObject("Part::Sphere", "Foliage2")
foliage2.Radius = 15
foliage2.Placement.Base = FreeCAD.Vector(15, 0, 40)
foliage3 = doc.addObject("Part::Sphere", "Foliage3")
foliage3.Radius = 15
foliage3.Placement.Base = FreeCAD.Vector(-15, 0, 40)

doc.recompute()

output_stl = os.path.abspath("model.stl")
output_obj = os.path.abspath("model.obj")
Mesh.export([trunk, foliage1, foliage2, foliage3], output_stl)
Mesh.export([trunk, foliage1, foliage2, foliage3], output_obj)
print("Exported to model.stl and model.obj")
"""

import re
with open("generated_freecad_script.py", "w") as f:
    # In a real scenario you would parse out just the python code block from generated_code.
    match = re.search(r'```python\n(.*?)\n```', generated_code, re.DOTALL)
    if match:
        f.write(match.group(1))
    elif generated_code and generated_code.strip().startswith("import"):
        f.write(generated_code)
    else:
        f.write(fallback_script)
        print("Using fallback script because generated code was invalid or missing.")

# Note: Small LLMs might generate incorrect FreeCAD Python API code.
# If `freecadcmd` fails in the next step, you can uncomment this to force the fallback script:
# with open("generated_freecad_script.py", "w") as f:
#     f.write(fallback_script)


## 4. Execute FreeCAD Headless
We run the generated script using `freecadcmd` to produce our 3D artifacts.

In [ ]:
!freecadcmd -c generated_freecad_script.py

## 5. Render Output inside Colab
Finally, we can load the exported `.stl` file using `trimesh` and render it right here in the notebook.

In [ ]:
import trimesh
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

mesh = trimesh.load("model.stl")
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

faces = mesh.faces
vertices = mesh.vertices

# Color the tree green
poly3d = Poly3DCollection(vertices[faces], facecolors='#228B22', alpha=0.9, linewidths=0)
ax.add_collection3d(poly3d)

# Scale the plot axes evenly
max_range = np.array([mesh.bounds[1][0]-mesh.bounds[0][0], 
                      mesh.bounds[1][1]-mesh.bounds[0][1], 
                      mesh.bounds[1][2]-mesh.bounds[0][2]]).max() / 2.0
mid_x = (mesh.bounds[1][0]+mesh.bounds[0][0]) * 0.5
mid_y = (mesh.bounds[1][1]+mesh.bounds[0][1]) * 0.5
mid_z = (mesh.bounds[1][2]+mesh.bounds[0][2]) * 0.5

ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

ax.grid(False)
plt.axis('off')
plt.title("Generated FreeCAD Tree Model")
plt.show()